# AI Engineering Buildcamp — Homework 1: RAG Mini-Project

This notebook works through the homework from **AI Engineering Buildcamp: From RAG to Agents — Lesson 1**.

The goal is to:

1. Download Allen Downey / Green Tea Press PDF books from `books.csv`
2. Convert PDFs to Markdown using `markitdown`
3. Prepare clean line-based documents
4. Chunk documents for RAG using `gitsource`
5. Index chunks with `minsearch`
6. Search the index
7. Build a simple RAG pipeline
8. Compare unstructured vs structured output token usage


In [2]:
# Uncomment this if you need to install packages in Jupyter
!uv add pandas requests markitdown gitsource minsearch openai pydantic

Resolved 238 packages in 897ms                                       
Checked 231 packages in 30ms                                         


In [3]:
from pathlib import Path
import re
import json
import requests
import pandas as pd

BASE_DIR = Path.cwd()
PDF_DIR = BASE_DIR / "books_pdf"
TEXT_DIR = BASE_DIR / "books_text"

PDF_DIR.mkdir(exist_ok=True)
TEXT_DIR.mkdir(exist_ok=True)

CSV_URL = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"

print("Base directory:", BASE_DIR)
print("PDF directory:", PDF_DIR)
print("Text directory:", TEXT_DIR)

Base directory: /Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG
PDF directory: /Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf
Text directory: /Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text


## 1. Download the books CSV

The homework gives a CSV file with the book titles and PDF links.


In [4]:
books_df = pd.read_csv(CSV_URL)
books_df

,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,http://greenteapress.com/thinkpython2/thinkpyt...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,http://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,http://greenteapress.com/complexity2/thinkcomp...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,http://greenteapress.com/thinkjava7/thinkjava2...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...
5,Think OS,https://greenteapress.com/wp/think-os/,http://greenteapress.com/thinkos/thinkos.pdf
6,Think C++,https://greenteapress.com/wp/think-c/,https://raw.githubusercontent.com/tscheffl/Thi...


## 2. Download all PDFs

This creates safe filenames from the book titles and downloads each PDF into `books_pdf/`.


In [6]:
def safe_filename(title: str, suffix: str = ".pdf") -> str:
    """Turn a book title into a safe filename."""
    name = re.sub(r"[^a-zA-Z0-9]+", "_", title).strip("_")
    return f"{name}{suffix}"


def download_pdf(url: str, output_path: Path) -> None:
    """Download a PDF file from a URL."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    output_path.write_bytes(response.content)


for _, row in books_df.iterrows():
    title = row["title"]
    pdf_url = row["pdf_url"]
    pdf_path = PDF_DIR / safe_filename(title)

    if pdf_path.exists():
        print(f"Already downloaded: {pdf_path.name}")
        continue

    print(f"Downloading {title}...")
    download_pdf(pdf_url, pdf_path)

print("Done.")

Done.


In [7]:
list(PDF_DIR.glob("*.pdf"))

[PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Think_OS.pdf'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Think_DSP.pdf'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Think_Java_2e.pdf'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Think_C.pdf'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Think_Complexity_2e.pdf'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Think_Python_2e.pdf'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_pdf/Physical_Modeling_in_MATLAB.pdf')]

## 3. Convert PDFs to Markdown

The homework asks us to use `markitdown`.

This cell uses the Python API, which works well inside a notebook.

In [14]:
!uv add 'markitdown[pdf]'

Resolved 238 packages in 4ms
Checked 231 packages in 30ms


In [10]:
from markitdown import MarkItDown

md_converter = MarkItDown()

for pdf_path in PDF_DIR.glob("*.pdf"):
    md_path = TEXT_DIR / pdf_path.with_suffix(".md").name

    if md_path.exists():
        print(f"Already converted: {md_path.name}")
        continue

    print(f"Converting {pdf_path.name}...")
    result = md_converter.convert(str(pdf_path))
    md_path.write_text(result.text_content, encoding="utf-8")

print("Done.")

Converting Think_OS.pdf...
Converting Think_DSP.pdf...
Converting Think_Java_2e.pdf...
Converting Think_C.pdf...
Converting Think_Complexity_2e.pdf...
Converting Think_Python_2e.pdf...
Converting Physical_Modeling_in_MATLAB.pdf...
Done.


In [11]:
list(TEXT_DIR.glob("*.md"))

[PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_Complexity_2e.md'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_DSP.md'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_C.md'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_Java_2e.md'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Physical_Modeling_in_MATLAB.md'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_OS.md'),
 PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_Python_2e.md')]

## Question 1 — Count lines in Think Python Markdown

Question: **How many lines are in the extracted content from the "Think Python" book?**

Options:

- 12,268
- 14,268
- 16,268
- 18,268


In [12]:
think_python_files = list(TEXT_DIR.glob("*Think_Python*.md"))
think_python_files

[PosixPath('/Users/davidalexander/Documents/AI_buildcamp/ai-engineering-buildcamp/homework2_RAG/books_text/Think_Python_2e.md')]

In [13]:
think_python_path = think_python_files[0]

content = think_python_path.read_text(encoding="utf-8")
line_count = len(content.splitlines())

print("File:", think_python_path.name)
print("Line count:", line_count)

File: Think_Python_2e.md
Line count: 16511


## 4. Prepare documents for chunking

- Read each Markdown file from `books_text/`
- Split the content into lines
- Remove empty lines and lines that contain only whitespace
- Turn each book into a dictionary with:
  - `source`: filename
  - `content`: list of non-empty lines


In [15]:
def prepare_book_documents(text_dir: Path):
    documents = []

    for md_path in sorted(text_dir.glob("*.md")):
        text = md_path.read_text(encoding="utf-8")
        lines = text.splitlines()

        # Remove empty or whitespace-only lines
        non_empty_lines = [line.strip() for line in lines if line.strip()]

        documents.append({
            "source": md_path.name,
            "content": non_empty_lines
        })

    return documents


book_documents = prepare_book_documents(TEXT_DIR)

print("Number of books:", len(book_documents))
print(book_documents[0].keys())
print(book_documents[0]["source"])
print("Non-empty lines in first book:", len(book_documents[0]["content"]))

Number of books: 7
dict_keys(['source', 'content'])
Physical_Modeling_in_MATLAB.md
Non-empty lines in first book: 5978


## 5. Chunk documents for RAG

The homework asks us to use the `gitsource` package and its `chunk_documents` function.

Parameters:

- `size=100`
- `step=50`

Each chunk uses a sliding window over the list of non-empty lines.


In [16]:
from gitsource import chunk_documents

chunks = chunk_documents(book_documents, size=100, step=50)

print("Total chunks:", len(chunks))
print("First chunk keys:", chunks[0].keys())
chunks[0]

Total chunks: 1009
First chunk keys: dict_keys(['start', 'content', 'source'])


{'start': 0,
 'content': ['| Physical | Modeling | in MATLAB |',
  '| -------- | -------- | --------- |',
  'Version 4.0',
  'Allen B. Downey',
  'Green Tea Press',
  'Needham, Massachusetts',
  'Physical Modeling in MATLAB',
  'Copyright 2012, 2019, 2021 Allen B. Downey',
  'Green Tea Press',
  '9 Washburn Ave',
  'Needham MA 02492',
  'Permissionisgrantedtocopy, distribute, and/ormodifythisdocumentunderthetermsofthe',
  'Creative Commons Attribution-NonCommercial 4.0 Unported License, which is available at',
  'https://greenteapress.com/matlab/license.',
  'This book was typeset by the author using pdflatex, among other free, open-source programs.',
  'The LaTeX source for this book is available from https://greenteapress.com/matlab.',
  'Theinformationinthisbookisdistributedonan“AsIs” basis, withoutwarranty. Whileevery',
  'precaution has been taken in the preparation of this work, neither the author nor No Starch',
  'Press, Inc. shall have any liability to any person or entity wit

## Question 2 — Chunks for Think Python

Question: **How many chunks are produced for the "Think Python" book with these settings?**

Options:

- 134
- 214
- 294
- 374


In [17]:
think_python_chunks = [
    chunk for chunk in chunks
    if "Think_Python" in chunk["source"]
]

print("Think Python chunks:", len(think_python_chunks))

Think Python chunks: 214


## 6. Prepare chunks for minsearch

`minsearch` expects each document to have text fields. The chunks currently have `content` as a list of lines, so we will join those lines into one string per chunk.


In [18]:
def prepare_documents(chunks):
    prepared = []

    for i, chunk in enumerate(chunks):
        prepared.append({
            "id": i,
            "source": chunk["source"],
            "content": "\n".join(chunk["content"])
        })

    return prepared


documents = prepare_documents(chunks)

print("Prepared documents:", len(documents))
documents[0]

Prepared documents: 1009


{'id': 0,
 'source': 'Physical_Modeling_in_MATLAB.md',
 'content': '| Physical | Modeling | in MATLAB |\n| -------- | -------- | --------- |\nVersion 4.0\nAllen B. Downey\nGreen Tea Press\nNeedham, Massachusetts\nPhysical Modeling in MATLAB\nCopyright 2012, 2019, 2021 Allen B. Downey\nGreen Tea Press\n9 Washburn Ave\nNeedham MA 02492\nPermissionisgrantedtocopy, distribute, and/ormodifythisdocumentunderthetermsofthe\nCreative Commons Attribution-NonCommercial 4.0 Unported License, which is available at\nhttps://greenteapress.com/matlab/license.\nThis book was typeset by the author using pdflatex, among other free, open-source programs.\nThe LaTeX source for this book is available from https://greenteapress.com/matlab.\nTheinformationinthisbookisdistributedonan“AsIs” basis, withoutwarranty. Whileevery\nprecaution has been taken in the preparation of this work, neither the author nor No Starch\nPress, Inc. shall have any liability to any person or entity with respect to any loss or damage

## Question 3 — Indexing with minsearch

Question: **How many documents/chunks did you index?**

Options:

- 719
- 919
- 1119
- 1319


In [19]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["source"]
)

index.fit(documents)

print("Indexed documents:", len(documents))

Indexed documents: 1009


## Question 4 — Search the index

Question: **Search for `"python function definition"`. Look at the top result. Which book did it come from?**

Options:

- Think Python
- Think DSP
- Think Java
- Think Complexity


In [20]:
results = index.search("python function definition", num_results=5)
results

[{'id': 833,
  'source': 'Think_Python_2e.md',
  'content': 'when you are comfortable with Python, I’ll make suggestions for installing Python on your\ncomputer.\nThere are a number of web pages you can use to run Python. If you already have a fa-\nvorite, go ahead and use it. Otherwise I recommend PythonAnywhere. I provide detailed\ninstructions for getting started at http://tinyurl.com/thinkpython2e.\nThere are two versions of Python, called Python 2 and Python 3. They are very similar, so\nif you learn one, it is easy to switch to the other. In fact, there are only a few differences you\nwill encounter as a beginner. This book is written for Python 3, but I include some notes\nabout Python 2.\nThe Python interpreter is a program that reads and executes Python code. Depending\non your environment, you might start the interpreter by clicking on an icon, or by typing\npython on a command line. When it starts, you should see output like this:\nPython 3.4.0 (default, Jun 19 2015, 14:20:2

In [21]:
top_result = results[0]

print("Top result source:", top_result["source"])
print()
print(top_result["content"][:1000])

Top result source: Think_Python_2e.md

when you are comfortable with Python, I’ll make suggestions for installing Python on your
computer.
There are a number of web pages you can use to run Python. If you already have a fa-
vorite, go ahead and use it. Otherwise I recommend PythonAnywhere. I provide detailed
instructions for getting started at http://tinyurl.com/thinkpython2e.
There are two versions of Python, called Python 2 and Python 3. They are very similar, so
if you learn one, it is easy to switch to the other. In fact, there are only a few differences you
will encounter as a beginner. This book is written for Python 3, but I include some notes
about Python 2.
The Python interpreter is a program that reads and executes Python code. Depending
on your environment, you might start the interpreter by clicking on an icon, or by typing
python on a command line. When it starts, you should see output like this:
Python 3.4.0 (default, Jun 19 2015, 14:20:21)
[GCC 4.8.2] on linux
Type "help

## 7. Full RAG

Now we build the RAG functions from the homework.


In [22]:
from openai import OpenAI

openai_client = OpenAI()

In [23]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)

    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()

    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## Question 5 — Run RAG

Question: **Do RAG for `"python function definition"`. What's the response?**

Then modify the functions to return the number of input and output tokens.

Question: **How many input tokens did we use for this one RAG query?**

Options:

- 4889
- 6889
- 8889
- 10889


In [24]:
query = "python function definition"

answer = rag(query)
print(answer)

A Python function is defined using the `def` keyword, followed by the function name and parentheses containing any parameters. The function's body is indented beneath the definition. Here’s a basic structure for defining a function in Python:

```python
def function_name(parameters):
    # Function body
    # Perform operations here
    return result  # Optional return statement
```

### Example
For example, let’s define a simple function that adds two numbers:

```python
def add_numbers(a, b):
    return a + b
```

In this example:
- `add_numbers` is the function name.
- `a` and `b` are parameters.
- The body of the function contains a return statement that provides the sum of `a` and `b`.

### Calling a Function
You can call the function using its name followed by arguments in parentheses:

```python
result = add_numbers(5, 3)  # result will contain the value 8
```

Functions allow you to create reusable code and help in organizing your program into manageable sections.


In [25]:
def llm_with_tokens(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens
    }


def rag_with_tokens(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    result = llm_with_tokens(prompt, instructions)
    return result


rag_result = rag_with_tokens("python function definition")

print("Input tokens:", rag_result["input_tokens"])
print("Output tokens:", rag_result["output_tokens"])
print("Total tokens:", rag_result["total_tokens"])
print()
print(rag_result["answer"])

Input tokens: 7311
Output tokens: 357
Total tokens: 7668

**Python Function Definition**

In Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. The general syntax is:

```python
def function_name(parameters):
    # function body
    # optional return statement
```

Here's a brief breakdown of the components:

1. **def**: This keyword indicates that you are defining a function.
2. **function_name**: This is the name you give to the function. It should be descriptive of what the function does.
3. **parameters**: These are optional inputs to the function. If the function does not require any input, you can leave the parentheses empty.
4. **function body**: This consists of the statements that define what the function does. It can include expressions, computations, and control flow.
5. **return**: This is optional but can be used to send back a value from the function to the caller.

### Example
Here's a simple exa

## 8. Structured output with Pydantic

Now we modify the RAG pipeline to return structured output.

The homework asks us to define a `RAGResponse` model.


In [26]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [27]:
def llm_structured_with_tokens(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return {
        "parsed": response.output_parsed,
        "raw_text": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens
    }


def rag_structured_with_tokens(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    result = llm_structured_with_tokens(prompt, instructions)
    return result


structured_result = rag_structured_with_tokens("python function definition")

print("Input tokens:", structured_result["input_tokens"])
print("Output tokens:", structured_result["output_tokens"])
print("Total tokens:", structured_result["total_tokens"])
print()
structured_result["parsed"]

Input tokens: 7497
Output tokens: 343
Total tokens: 7840



RAGResponse(answer="In Python, a function is defined using the `def` keyword, followed by the function name and parentheses containing optional parameters. The body of the function contains a sequence of statements that define what the function does. Here's a basic outline of a function definition:\n\n```python\ndef function_name(parameters):\n    # function body\n    # return statement (optional)\n```\n\n### Example:\nHere’s a simple example function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    return a + b\n```\n\nIn this example:\n- `add_numbers` is the function name.\n- `a` and `b` are parameters (inputs).\n- The function returns the sum of `a` and `b`.\n\n### Calling a Function:\nTo use the function, you would call it by its name and provide the necessary arguments:\n\n```python\nresult = add_numbers(3, 5)\nprint(result)  # Output: 8\n```\n\nThis prints `8`, which is the result of adding `3` and `5`. \n\nFunctions help in organizing code into reusable pieces, r

## Question 6 — Structured vs unstructured input tokens

Question: **How many MORE input tokens does the structured output version use compared to the unstructured version?**

Options:

- 24
- 224
- 424
- 624


In [28]:
unstructured_input_tokens = rag_result["input_tokens"]
structured_input_tokens = structured_result["input_tokens"]

difference = structured_input_tokens - unstructured_input_tokens

print("Unstructured input tokens:", unstructured_input_tokens)
print("Structured input tokens:", structured_input_tokens)
print("Difference:", difference)

Unstructured input tokens: 7311
Structured input tokens: 7497
Difference: 186


## 9. Final answers helper


In [29]:
answers = {
    "q1_think_python_line_count": line_count,
    "q2_think_python_chunk_count": len(think_python_chunks),
    "q3_indexed_documents": len(documents),
    "q4_top_search_result_source": top_result["source"],
    "q5_unstructured_input_tokens": rag_result["input_tokens"],
    "q6_structured_extra_input_tokens": difference,
}

answers

{'q1_think_python_line_count': 16511,
 'q2_think_python_chunk_count': 214,
 'q3_indexed_documents': 1009,
 'q4_top_search_result_source': 'Think_Python_2e.md',
 'q5_unstructured_input_tokens': 7311,
 'q6_structured_extra_input_tokens': 186}